# GPU Portfolio Bench — Cloud Sweep

**Runtime:** GPU → T4 (free) or A100/V100 (Colab Pro)  
**How to use:** Runtime → Run all. Results download automatically at the end.

Benchmarks three backends:
- **NumPy CPU** — baseline
- **PyTorch CUDA** — batched tensor ops on GPU
- **CuPy raw kernel** — hand-written CUDA kernel, one thread per path

Sweeps `n_assets ∈ {10, 50, 100}` × `n_paths ∈ {10k, 100k, 1M, 10M}`.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Runtime → Change runtime type → GPU")

gpu_name     = torch.cuda.get_device_name(0)
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {gpu_total_gb:.1f} GB")
print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}")

def free_vram_gb() -> float:
    """Bytes of free GPU memory right now."""
    torch.cuda.synchronize()
    free, _ = torch.cuda.mem_get_info()
    return free / 1e9

def safe_chunk_size(n_assets: int, dtype_bytes: int = 4, headroom: float = 0.5) -> int:
    """
    Largest n_paths chunk that fits in GPU memory.
    Reserves `headroom` fraction of free VRAM for other tensors (cov, L, w, etc.).
    Each chunk needs 2 × n_paths × n_assets × dtype_bytes bytes (z + correlated).
    """
    free = free_vram_gb() * 1e9 * headroom
    chunk = int(free / (2 * n_assets * dtype_bytes))
    return max(1_000, min(chunk, 5_000_000))  # floor 1k, cap 5M

In [ ]:
%%capture
# Install extra deps not pre-installed on Colab
!pip install -q yfinance pyarrow cvxpy pynvml tqdm
# CuPy matching Colab's CUDA 12.x
!pip install -q cupy-cuda12x

In [ ]:
import time
import csv
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yfinance as yf

try:
    import cupy as cp
    CUPY_AVAILABLE = True
    print(f"CuPy {cp.__version__} — raw CUDA kernel enabled")
except ImportError:
    CUPY_AVAILABLE = False
    print("CuPy not available")

try:
    import pynvml
    pynvml.nvmlInit()
    PYNVML_OK = True
except Exception:
    PYNVML_OK = False

RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "benchmark_sweep.csv"

warnings.filterwarnings("ignore")
print("Imports done.")

In [ ]:
SP500_SUBSET = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "BRK-B", "UNH", "JNJ",
    "JPM", "V",    "PG",    "MA",   "HD",   "CVX",  "MRK",  "ABBV",  "PEP",  "KO",
    "AVGO","COST", "WMT",   "MCD",  "BAC",  "DIS",  "CSCO", "ACN",   "TMO",  "ABT",
    "LIN", "DHR",  "VZ",    "ADBE", "CRM",  "NFLX", "TXN",  "NEE",   "WFC",  "PM",
    "AMD", "RTX",  "QCOM",  "HON",  "UPS",  "AMGN", "LOW",  "SBUX",  "IBM",  "GS",
    # Extended to hit n_assets=100
    "SPGI","CAT",  "BLK",   "DE",   "MMM",  "MO",   "GE",   "MDLZ",  "ADP",  "ISRG",
    "ADI", "TGT",  "SYK",   "ZTS",  "CB",   "PLD",  "AMT",  "SO",    "DUK",  "CL",
    "EOG", "SLB",  "MPC",   "PSA",  "CME",  "ITW",  "NOC",  "ETN",   "APD",  "EW",
    "REGN","HCA",  "GD",    "MSI",  "MCO",  "ORLY", "AZO",  "AFL",   "TRV",  "ICE",
    "AEP", "FCX",  "SRE",   "NSC",  "WM",   "ECL",  "EXC",  "XEL",   "FIS",  "KMB",
]

print(f"Fetching prices for {len(SP500_SUBSET)} tickers (2018–present)...")
raw = yf.download(SP500_SUBSET, start="2018-01-01", auto_adjust=True, progress=True)
prices = raw["Close"].dropna(axis=1, how="all")
returns = np.log(prices / prices.shift(1)).dropna()
print(f"Returns shape: {returns.shape}  ({returns.shape[1]} clean tickers)")

In [ ]:
# ── CPU model ────────────────────────────────────────────────────────────────
def compute_var_cvar_cpu(returns_np, weights, n_paths=100_000, confidence=0.95, seed=42):
    rng = np.random.default_rng(seed)
    mu  = returns_np.mean(axis=0)
    cov = np.cov(returns_np.T) + np.eye(returns_np.shape[1]) * 1e-8
    L   = np.linalg.cholesky(cov)
    dt  = 1 / 252.0
    z   = rng.standard_normal((n_paths, returns_np.shape[1]))
    sim = mu * dt + (z @ L.T) * np.sqrt(dt)
    port = sim @ weights
    t0 = time.perf_counter(); port = sim @ weights; elapsed = time.perf_counter() - t0
    # Redo with timing around the whole thing
    t0 = time.perf_counter()
    z   = rng.standard_normal((n_paths, returns_np.shape[1]))
    sim = mu * dt + (z @ L.T) * np.sqrt(dt)
    port = sim @ weights
    elapsed = time.perf_counter() - t0
    thr = np.percentile(port, (1 - confidence) * 100)
    tail = port[port <= thr]
    return {"device": "cpu", "n_paths": n_paths,
            "VaR": float(-thr), "CVaR": float(-tail.mean()) if len(tail) else float(-thr),
            "elapsed_sec": elapsed, "throughput_paths_per_sec": n_paths / elapsed,
            "gpu_mem_mb": 0.0, "gpu_util_pct": -1.0}

In [ ]:
# ── PyTorch CUDA model (chunked to avoid OOM) ────────────────────────────────
def compute_var_cvar_gpu(returns_np, weights, n_paths=100_000, confidence=0.95, seed=42):
    dev = torch.device("cuda")
    torch.manual_seed(seed)

    mu  = torch.tensor(returns_np.mean(axis=0), dtype=torch.float32, device=dev)
    cov = torch.tensor(np.cov(returns_np.T) + np.eye(returns_np.shape[1]) * 1e-8,
                       dtype=torch.float32, device=dev)
    w   = torch.tensor(weights, dtype=torch.float32, device=dev)
    L   = torch.linalg.cholesky(cov)
    dt  = 1 / 252.0

    # Warm-up with a tiny batch to compile kernels before timing
    _ = (torch.randn(256, returns_np.shape[1], device=dev) @ L.T) @ w
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    chunk = safe_chunk_size(returns_np.shape[1], dtype_bytes=4)
    n_chunks = (n_paths + chunk - 1) // chunk
    all_port = []

    t0 = time.perf_counter()
    rng = torch.Generator(device=dev)
    rng.manual_seed(seed)
    for i in range(n_chunks):
        size = min(chunk, n_paths - i * chunk)
        z    = torch.randn(size, returns_np.shape[1], device=dev, generator=rng)
        sim  = mu * dt + (z @ L.T) * (dt ** 0.5)
        all_port.append((sim @ w).cpu())   # move to CPU immediately to free VRAM
        del z, sim
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    port = torch.cat(all_port)
    thr  = torch.quantile(port, 1 - confidence).item()
    tail = port[port <= thr]
    mem  = torch.cuda.max_memory_allocated() / 1e6
    torch.cuda.reset_peak_memory_stats()

    gpu_util = -1.0
    if PYNVML_OK:
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        gpu_util = float(pynvml.nvmlDeviceGetUtilizationRates(h).gpu)

    chunks_used = f" ({n_chunks} chunk{'s' if n_chunks>1 else ''})" if n_chunks > 1 else ""
    return {"device": f"cuda{chunks_used}", "n_paths": n_paths,
            "VaR": float(-thr), "CVaR": float(-tail.mean().item()) if tail.numel() else float(-thr),
            "elapsed_sec": elapsed, "throughput_paths_per_sec": n_paths / elapsed,
            "gpu_mem_mb": mem, "gpu_util_pct": gpu_util}

In [ ]:
# ── CuPy raw CUDA kernel ─────────────────────────────────────────────────────
_GBM_KERNEL_CODE = r"""
#include <curand_kernel.h>
extern "C" __global__
void gbm_portfolio_kernel(
    const double* __restrict__ mu,
    const double* __restrict__ chol,
    const double* __restrict__ weights,
    double* __restrict__ out,
    const int n_assets,
    const double dt,
    const unsigned long long seed,
    const int n_paths
) {
    int path_id = blockIdx.x * blockDim.x + threadIdx.x;
    if (path_id >= n_paths) return;
    curandState_t state;
    curand_init(seed, (unsigned long long)path_id, 0, &state);
    double portfolio_return = 0.0;
    for (int a = 0; a < n_assets; a++) {
        double correlated = 0.0;
        for (int j = 0; j <= a; j++) {
            double z = curand_normal_double(&state);
            correlated += chol[a * n_assets + j] * z;
        }
        double asset_return = mu[a] * dt + correlated * sqrt(dt);
        portfolio_return += weights[a] * asset_return;
    }
    out[path_id] = portfolio_return;
}
"""

import os
_CUDA_HOME = os.environ.get("CUDA_HOME", "/usr/local/cuda")
_CUPY_KERNEL = None

def _get_kernel():
    global _CUPY_KERNEL
    if _CUPY_KERNEL is None:
        print(f"Compiling CuPy CUDA kernel (CUDA headers: {_CUDA_HOME}/include)...")
        _CUPY_KERNEL = cp.RawKernel(
            _GBM_KERNEL_CODE,
            "gbm_portfolio_kernel",
            options=("--std=c++14", f"-I{_CUDA_HOME}/include"),
            backend="nvrtc",
        )
        print("Kernel compiled OK.")
    return _CUPY_KERNEL

def compute_var_cvar_cupy(returns_np, weights, n_paths=100_000, confidence=0.95, seed=42):
    if not CUPY_AVAILABLE:
        raise RuntimeError("CuPy not available")
    kernel = _get_kernel()
    mu       = returns_np.mean(axis=0)
    cov      = np.cov(returns_np.T) + np.eye(returns_np.shape[1]) * 1e-8
    L        = np.linalg.cholesky(cov)
    dt       = 1 / 252.0
    mu_gpu   = cp.asarray(mu,      dtype=cp.float64)
    chol_gpu = cp.asarray(L,       dtype=cp.float64)
    w_gpu    = cp.asarray(weights, dtype=cp.float64)
    out_gpu  = cp.empty(n_paths,   dtype=cp.float64)
    # Warm-up
    out_wup = cp.empty(1000, dtype=cp.float64)
    kernel((4,), (256,), (mu_gpu, chol_gpu, w_gpu, out_wup,
                          returns_np.shape[1], dt, np.uint64(0), 1000))
    cp.cuda.Stream.null.synchronize()
    threads = 256
    blocks  = (n_paths + threads - 1) // threads
    t0 = time.perf_counter()
    kernel((blocks,), (threads,), (mu_gpu, chol_gpu, w_gpu, out_gpu,
                                   returns_np.shape[1], dt, np.uint64(seed), n_paths))
    cp.cuda.Stream.null.synchronize()
    elapsed = time.perf_counter() - t0
    thr  = float(cp.percentile(out_gpu, (1 - confidence) * 100).get())
    tail = out_gpu[out_gpu <= thr]
    mem  = cp.get_default_memory_pool().used_bytes() / 1e6
    gpu_util = -1.0
    if PYNVML_OK:
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        gpu_util = float(pynvml.nvmlDeviceGetUtilizationRates(h).gpu)
    return {"device": "cupy_kernel", "n_paths": n_paths,
            "VaR": float(-thr), "CVaR": float(-tail.mean().get()) if tail.size > 0 else float(-thr),
            "elapsed_sec": elapsed, "throughput_paths_per_sec": n_paths / elapsed,
            "gpu_mem_mb": mem, "gpu_util_pct": gpu_util}

print("CuPy kernel cell loaded.")

In [ ]:
ASSET_COUNTS = [10, 50, 100]
PATH_COUNTS  = [10_000, 100_000, 1_000_000, 10_000_000]
DEVICES      = ["cpu", "cuda"] + (["cupy_kernel"] if CUPY_AVAILABLE else [])

all_assets = list(returns.columns)
rows = []
total = sum(
    1 for n in ASSET_COUNTS if len(all_assets) >= n
    for _ in PATH_COUNTS for _ in DEVICES
)
idx = 0

for n_assets in ASSET_COUNTS:
    if len(all_assets) < n_assets:
        print(f"  Only {len(all_assets)} tickers — skipping n_assets={n_assets}")
        continue
    assets = all_assets[:n_assets]
    R = returns[assets].to_numpy(dtype=np.float64)
    w = np.ones(n_assets) / n_assets

    for n_paths in PATH_COUNTS:
        for device in DEVICES:
            idx += 1
            label = f"[{idx}/{total}] {device:12s}  n_assets={n_assets:3d}  n_paths={n_paths:>10,}"
            try:
                if device == "cpu":
                    result = compute_var_cvar_cpu(R, w, n_paths=n_paths)
                elif device == "cuda":
                    result = compute_var_cvar_gpu(R, w, n_paths=n_paths)
                    result["device"] = "cuda"
                else:
                    result = compute_var_cvar_cupy(R, w, n_paths=n_paths)

                rows.append({
                    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
                    "device": result["device"],
                    "n_assets": n_assets,
                    "n_paths": n_paths,
                    "VaR":   result["VaR"],
                    "CVaR":  result["CVaR"],
                    "elapsed_sec": result["elapsed_sec"],
                    "throughput_paths_per_sec": result["throughput_paths_per_sec"],
                    "gpu_mem_mb":   result["gpu_mem_mb"],
                    "gpu_util_pct": result["gpu_util_pct"],
                })
                tput = result["throughput_paths_per_sec"]
                tput_str = f"{tput/1e6:.1f}M/s" if tput >= 1e6 else f"{tput/1e3:.0f}k/s"
                extra = f"  {result['gpu_mem_mb']:.0f} MB VRAM" if device != "cpu" else ""
                print(f"{label}  →  {result['elapsed_sec']:.3f}s  {tput_str}{extra}")

            except (RuntimeError, MemoryError) as e:
                if device != "cpu":
                    torch.cuda.empty_cache()
                print(f"{label}  SKIPPED: {e}")

print(f"\nSweep done — {len(rows)} / {total} configs completed.")

# ── Speedup summary ───────────────────────────────────────────────────────────
df = pd.DataFrame(rows)

if not df.empty:
    cpu_df  = df[df.device == "cpu"][["n_assets","n_paths","throughput_paths_per_sec"]].rename(columns={"throughput_paths_per_sec": "cpu_tput"})
    cuda_df = df[df.device == "cuda"][["n_assets","n_paths","throughput_paths_per_sec"]].rename(columns={"throughput_paths_per_sec": "cuda_tput"})
    summary = pd.merge(cpu_df, cuda_df, on=["n_assets","n_paths"], how="inner")
    summary["speedup_cuda"] = summary["cuda_tput"] / summary["cpu_tput"]

    if CUPY_AVAILABLE and "cupy_kernel" in df.device.values:
        cupy_df = df[df.device == "cupy_kernel"][["n_assets","n_paths","throughput_paths_per_sec"]].rename(columns={"throughput_paths_per_sec": "cupy_tput"})
        summary = pd.merge(summary, cupy_df, on=["n_assets","n_paths"], how="left")
        summary["speedup_cupy"] = summary["cupy_tput"] / summary["cpu_tput"]
    else:
        summary["cupy_tput"]    = float("nan")
        summary["speedup_cupy"] = float("nan")

    def fmt(x):
        return f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}k"

    print("\n── Throughput & Speedup ─────────────────────────────────────────")
    for _, r in summary.iterrows():
        cupy_col = summary.get("cupy_tput")
        has_cupy = cupy_col is not None and not pd.isna(r.get("cupy_tput", float("nan")))
        line = (f"  {int(r.n_assets):3d} assets × {int(r.n_paths)/1e6:4.1f}M paths"
                f"  CPU={fmt(r.cpu_tput)}/s"
                f"  CUDA={fmt(r.cuda_tput)}/s  ({r.speedup_cuda:.1f}×)")
        if has_cupy:
            line += f"  CuPy={fmt(r.cupy_tput)}/s  ({r.speedup_cupy:.1f}×)"
        print(line)
else:
    print("No results to summarise — all configs were skipped.")

# ── Save & download ───────────────────────────────────────────────────────────
df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved {len(df)} rows → {RESULTS_PATH}")

try:
    from google.colab import files
    files.download(str(RESULTS_PATH))
    print("Download triggered — check your browser's downloads folder.")
except ImportError:
    print(f"Not in Colab — file is at: {RESULTS_PATH}")